In [ ]:
import itertools
import os
import time
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib import rcParams
from any_functions import create_data_folder
from scripts.write_arrays_to_excel import write_arrays_excel
from scripts.write_arrays_to_txt import write_arrays_txt
from scripts.create_map_and_save import create_map_and_save


# импорт настроек
from config import *
# ===================
# импорт драйверов
# ===================

# from devices.yokogawa.OSA_Yokogawa_new import OSA_Yokogawa_new
# from devices.btf_100.btf_100 import BTF100
from devices.rsa_device.rf_measure_lib import rf_measurement
from devices.rsa_device.RF306B import RF306B
from devices.odl_650.OpticDelayLine_new import OpticDelayLine
from devices.ut_3005.VoltageDriverUT3005 import VoltageDriverUT3005
from devices.cdl_1015.CLD1015 import CLD1015




# ============================================================
# Функция для построение 6 карт для одного напряжения
# ============================================================


def _build_voltage_maps(voltage, main_folder, data_buf):
    """Внутренняя функция: строит 6 тепловых карт для заданного напряжения."""
    from pathlib import Path
    import numpy as np
    
    # Фильтруем данные по напряжению
    mask = np.array(data_buf['voltage']) == voltage
    
    # Папка для карт этого напряжения
    plot_subfolder = Path(main_folder) / f"voltage_{voltage}V" / "maps"
    plot_subfolder.mkdir(parents=True, exist_ok=True)
    
    # Данные для осей
    x = np.array(data_buf['current'])[mask]
    y = np.array(data_buf['delay'])[mask]
    
    # === Конфигурация карт (6 карт: freq + amplitude для каждого span) ===
    # rf_freq_* в Hz, нужно перевести в GHz (делим на 1e+9)
    maps_config = [
        (np.array(data_buf['rf_freq_max']) / 1e+9, f'rf_freq_{max_span}', 'Frequency, GHz', max_span),
        (data_buf['rf_peak_amplitude_max'], f'rf_peak_amplitude_{max_span}', 'Peak Amplitude, dBm', max_span),
        # (np.array(data_buf['rf_freq_mid']) / 1e+9, f'rf_freq_{middle_span}', 'Frequency, GHz', middle_span),
        # (data_buf['rf_peak_amplitude_mid'], f'rf_peak_amplitude_{middle_span}', 'Peak Amplitude, dBm', middle_span),
        # (np.array(data_buf['rf_freq_min']) / 1e+9, f'rf_freq_{min_span}', 'Frequency, GHz', min_span),
        # (data_buf['rf_peak_amplitude_min'], f'rf_peak_amplitude_{min_span}', 'Peak Amplitude, dBm', min_span),
    ]
    
    for z_raw, fname_suffix, z_label, span_label in maps_config:
        z = np.array(z_raw)[mask]
        create_map_and_save(
            x_arr=[x.tolist(), "Current, mA"],
            y_arr=[y.tolist(), "Delay, ps"],
            z_arr=[z.tolist(), z_label],
            title=f"Voltage {voltage}V - {z_label} ({span_label})",
            folder_path=plot_subfolder,
            filename=f"{fname_suffix}",
            show_plot=False  
        )


def main():
    main_folder = create_data_folder(prefix='laser_with_btf')
    
    params = itertools.product(VOLTAGES, CURRENTS, DELAYS)

    # === Буфер для сбора данных ===
    collected_data = {
        'voltage': [], 'current': [], 'delay': [],
        'rf_freq_max': [], 'rf_peak_amplitude_max': [],
        # 'rf_freq_mid': [], 'rf_peak_amplitude_mid': [],
        # 'rf_freq_min': [], 'rf_peak_amplitude_min': [],
    }
    
    current_voltage_batch = None
    
    voltage_prev = -1
    current_prev = -1
    delay_prev = -1
    
    try:
        # ========================
        # ИНИЦИАЛИЗАЦИЯ ПРИБОРОВ
        # ========================
        odl = OpticDelayLine('COM10')
        odl.initialize()
        # osa = OSA_Yokogawa_new()
        rf_device = RF306B()
        LD = CLD1015()
        LD.turn_on_all()

        ut = VoltageDriverUT3005('COM8')
        ut.turn_on()

        for idx, (voltage, current, delay) in enumerate(params, 1):

            # === Построение карт при смене напряжения ===
            if current_voltage_batch is not None and voltage != current_voltage_batch:
                _build_voltage_maps(current_voltage_batch, main_folder, collected_data)
            current_voltage_batch = voltage
            
            

            # === НАСТРОЙКА ОБОРУДОВАНИЯ ===
            
            voltage_next = voltage
            if voltage_prev != voltage_next:
                ut.set_voltage(voltage=voltage)
                voltage_prev = voltage_next
            
            current_next = current
            if current_prev != current_next:
                LD.set_current(current=current)
                current_prev = current_next
            
            delay_next = delay
            if delay_prev != delay_next:
                odl.set_time_delay(time_delay=delay)
                delay_prev = delay_next
            
            time.sleep(3)
            

            # === ИЗМЕРЕНИЯ ===
            
            # === Имена файлов (единицы измерения как есть: mA, V, ps, Hz) ===
            # rf_spectrum_delay_Xps_current_XmA_span_XXXHz_voltage_XV
            
            
            # === Каждый спан в своей папке ===
            


            # === Заполнение буфера для карт ===
            collected_data['voltage'].append(voltage)
            collected_data['current'].append(current)
            collected_data['delay'].append(delay)
            collected_data['rf_freq_max'].append(rf_freq_max)
            collected_data['rf_peak_amplitude_max'].append(rf_amplitude_max)
            # collected_data['rf_freq_mid'].append(rf_freq_mid)
            # collected_data['rf_peak_amplitude_mid'].append(rf_amplitude_mid)
            # collected_data['rf_freq_min'].append(rf_freq_min)
            # collected_data['rf_peak_amplitude_min'].append(rf_amplitude_min)
            

        
        # === Построение карт для ПОСЛЕДНЕГО напряжения ===
        if current_voltage_batch is not None:
            _build_voltage_maps(current_voltage_batch, main_folder, collected_data)

        
    finally:
        try:
            odl.disconnect()
            LD.turn_off_all()
            ut.out_off_and_close_COM()
        except Exception as e:
            print(f"Ошибка при отключении: {e}")    
        
    

if __name__ == "__main__":
    main()

Порт COM10 открыт
API Version 3.11.0047.0
One device found.
Device type: RSA306B
Device serial number: B032586
RF306B inited!
CLD1015 inited!
tec is ON
laser is ON
Voltage Driver is connected
The output is on
Set voltage 0V
Set laser current 100mA
Установка задержки: 0
Отправлено: T0
Получено: T0
Ответ не 'Done'. Попытка 1 из 10
Пауза 3.0 сек перед повторной отправкой...
Отправлено: T0
Получено: T0
Получено: Done
Задержка установлена успешно (попытка 2)
Данные успешно сохранены
Установка задержки: 10
Отправлено: T10
Получено: T10
Получено: Done
Задержка установлена успешно (попытка 1)
Данные успешно сохранены
Установка задержки: 20
Отправлено: T20
Получено: T20
Получено: Done
Задержка установлена успешно (попытка 1)
Данные успешно сохранены
Установка задержки: 30
Отправлено: T30
Получено: T30
Ответ не 'Done'. Попытка 1 из 10
Пауза 3.0 сек перед повторной отправкой...
Отправлено: T30
Получено: T30
Получено: Done
Задержка установлена успешно (попытка 2)
Данные успешно сохранены
Установка